# Sprint 4 — Faithfulness evaluation (DeepEval)

Runs every golden question through the live RAG pipeline and scores each answer with DeepEval's `FaithfulnessMetric`.

**What faithfulness measures:** the judge splits the answer into claims and checks each one against the chunks the pipeline retrieved. Score = supported claims ÷ total claims (0–1). A low score means the answer said things the retrieved context does not back up. It does **not** mean the retriever picked the wrong chunks; that is Contextual Relevancy, measured separately.

**Before running:**
1. Set `OPEN_AI_API_KEY` (or `OPENAI_API_KEY`) in `.env`, and have the persisted ChromaDB in `vectorstore/`.
2. Fill in the golden questions in section 3.
3. Run top to bottom. Every question costs one pipeline call plus a few judge calls.

## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import sys

from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# The project's .env uses OPEN_AI_API_KEY, but DeepEval only reads OPENAI_API_KEY.
load_dotenv(ROOT / ".env", override=True)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPEN_AI_API_KEY", "")
if not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("No OpenAI API key found. Set OPEN_AI_API_KEY in .env.")

from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

from src.pipeline import run_query

## 2. Evaluation config

Evaluation-only values. The pipeline's own answering model still comes from `src/config/settings.py`; only the judge is set here.

In [ ]:
JUDGE_MODEL = "gpt-4.1"   # LLM-as-a-judge; deliberately not the answering model
THRESHOLD = 0.9           # a score at or above this counts as a pass

RESULTS_PATH = ROOT / "tests" / "eval" / "faithfulness_results.md"

## 3. Golden questions

Replace `"question"` with each golden question. Slots left as `"question"` are skipped.

In [ ]:
Q1 = ["question"]
Q2 = ["question"]
Q3 = ["question"]
Q4 = ["question"]
Q5 = ["question"]
Q6 = ["question"]
Q7 = ["question"]
Q8 = ["question"]
Q9 = ["question"]
Q10 = ["question"]
Q11 = ["question"]
Q12 = ["question"]
Q13 = ["question"]
Q14 = ["question"]
Q15 = ["question"]

GOLDEN_QUESTIONS = [
    (f"Q{i}", q[0])
    for i, q in enumerate((Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q8, Q9, Q10, Q11, Q12, Q13, Q14, Q15), start=1)
    if q[0].strip() and q[0] != "question"
]
print(f"{len(GOLDEN_QUESTIONS)} golden question(s) loaded")

## 4. Run each question through the pipeline

In [ ]:
runs = []
for qid, question in GOLDEN_QUESTIONS:
    result = run_query(question)
    runs.append({
        "id": qid,
        "question": question,
        "answer": result["answer"],
        "status": result["status"],
        "latency_seconds": result["latency_seconds"],
        "retrieval_context": [chunk["text"] for chunk in result["chunks"]],
    })
    print(f"{qid}: status={result['status']}, {len(result['chunks'])} chunks, {result['latency_seconds']}s")

## 5. Score faithfulness

Answers with no retrieved context (guardrail refusals, no results, errors) are marked *not scored*. There is nothing to be faithful to, so a 0 or 1 would be misleading.

In [ ]:
rows = []
for run in runs:
    row = {
        "ID": run["id"],
        "Question": run["question"],
        "Status": run["status"],
        "Score": None,
        "Pass": None,
        "Reason": "",
        "Latency (s)": run["latency_seconds"],
    }

    if run["status"] != "ok" or not run["retrieval_context"]:
        row["Reason"] = f"Not scored: pipeline status '{run['status']}', no retrieved context"
        print(f"\n{run['id']}: NOT SCORED ({run['status']})")
        rows.append(row)
        continue

    metric = FaithfulnessMetric(
        threshold=THRESHOLD,
        model=JUDGE_MODEL,
        include_reason=True,
        async_mode=False,  # avoids event-loop clashes inside Jupyter
    )
    metric.measure(LLMTestCase(
        input=run["question"],
        actual_output=run["answer"],
        retrieval_context=run["retrieval_context"],
    ))

    row["Score"] = round(metric.score, 3) # type: ignore
    row["Pass"] = metric.is_successful()
    row["Reason"] = metric.reason
    rows.append(row)

    print(f"\n{run['id']}: faithfulness={row['Score']} ({'PASS' if row['Pass'] else 'FAIL'})")
    print(f"  Q: {run['question']}")
    print(f"  Reason: {metric.reason}")

## 6. Summary

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(rows)
scored = df[df["Score"].notna()]

display(df[["ID", "Question", "Status", "Score", "Pass", "Latency (s)"]])

if len(scored):
    print(f"Scored: {len(scored)}/{len(df)}")
    print(f"Mean faithfulness: {scored['Score'].mean():.3f}")
    print(f"Pass rate (>= {THRESHOLD}): {int(scored['Pass'].sum())}/{len(scored)}")
else:
    print("No questions were scored.")

## 7. Save results

In [ ]:
# def _cell(value):
#     return str(value).replace("|", "\\|").replace("\n", " ")

# lines = [
#     "# Faithfulness results",
#     "",
#     f"- Run: {datetime.now():%Y-%m-%d %H:%M}",
#     f"- Judge model: `{JUDGE_MODEL}`",
#     f"- Threshold: {THRESHOLD}",
# ]
# if len(scored):
#     lines += [
#         f"- Mean faithfulness: {scored['Score'].mean():.3f} over {len(scored)} scored question(s)",
#         f"- Pass rate: {int(scored['Pass'].sum())}/{len(scored)}",
#     ]
# lines += [
#     "",
#     "| ID | Question | Status | Score | Pass | Latency (s) | Reason |",
#     "|---|---|---|---|---|---|---|",
# ]
# for row in rows:
#     score = "–" if row["Score"] is None else row["Score"]
#     passed = "–" if row["Pass"] is None else ("Yes" if row["Pass"] else "No")
#     lines.append(
#         f"| {row['ID']} | {_cell(row['Question'])} | {row['Status']} | {score} | {passed} "
#         f"| {row['Latency (s)']} | {_cell(row['Reason'])} |"
#     )

# RESULTS_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
# print(f"Wrote {RESULTS_PATH.relative_to(ROOT)}")